# 01_深度研究 (Deep Research Agent)

**来源:** [LangChain Docs — Build a deep research agent](https://docs.langchain.com/deep-agents/deep-research)

本教程演示如何从头开始构建一个多步骤的 Web 研究智能体。该智能体将研究问题分解为聚焦的子任务，委派给专门的子智能体，并将发现综合成一份带有引用的综合报告。

## 1. 核心概念

| 概念 | 说明 |
|------|------|
| 子智能体 (Subagents) | 用于并行、上下文隔离的研究任务 |
| 自定义工具 | 使用 Tavily 进行 Web 搜索 |
| 多步规划 | 内置 `planning` 工具用于任务分解 |
| 编排工作流 | 主智能体协调多个子智能体并综合结果 |

## 2. 环境准备

安装依赖并设置 API 密钥。

In [ ]:
# 安装核心依赖
# pip install deepagents tavily-python httpx markdownify langchain-anthropic langchain-core

# 设置 API 密钥（按需选择模型提供商）
import os

# os.environ["ANTHROPIC_API_KEY"] = "your-api-key"  # Claude
# os.environ["GOOGLE_API_KEY"] = "your-api-key"    # Gemini
# os.environ["TAVILY_API_KEY"] = "your-api-key"    # Tavily 搜索
# os.environ["LANGSMITH_API_KEY"] = "your-api-key" # 可选：LangSmith 追踪

print("环境准备完成")

## 3. 定义搜索工具

使用 Tavily 搜索 API 获取网页链接，然后抓取完整网页内容供智能体分析。

In [ ]:
# 导入必要的库
import os
from typing import Annotated, Literal

import httpx
from langchain.tools import InjectedToolArg, tool
from markdownify import markdownify
from tavily import TavilyClient

# 初始化 Tavily 客户端
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """获取网页内容并转换为 Markdown 格式"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"获取 {url} 失败: {e!s}"


@tool(parse_docstring=True)
def tavily_search(
    query: str,
    max_results: Annotated[int, InjectedToolArg] = 1,
    topic: Annotated[
        Literal["general", "news", "finance"], InjectedToolArg
    ] = "general",
) -> str:
    """搜索 Web 获取指定查询的信息。

    使用 Tavily 发现相关 URL，然后获取并返回完整的网页内容（Markdown 格式）。

    参数:
        query: 要执行的搜索查询
        max_results: 返回的最大结果数（默认: 1）
        topic: 主题过滤器 - 'general', 'news', 或 'finance'（默认: 'general'）

    返回:
        带有完整网页内容的格式化搜索结果
    """
    search_results = tavily_client.search(
        query,
        max_results=max_results,
        topic=topic,
    )
    result_texts = []
    for result in search_results.get("results", []):
        url = result["url"]
        title = result["title"]
        content = fetch_webpage_content(url)
        result_texts.append(f"## {title}\n**URL:** {url}\n\n{content}\n---")

    return f"找到 {len(result_texts)} 条 '{query}' 的结果:\n\n" + "\n".join(
        result_texts
    )


print("搜索工具定义完成")

## 4. 定义提示模板

研究智能体需要三个核心提示模板：研究工作流指令、研究员指令、子智能体委派指令。

In [ ]:
# 编排工作流指令：定义主智能体的研究流程
RESEARCH_WORKFLOW_INSTRUCTIONS = """# Research Workflow

Follow this workflow for all research requests:

1. **Plan**: Create a todo list with write_todos to break down the research into focused tasks
2. **Save the request**: Use write_file() to save the user's research question to `/research_request.md`
3. **Research**: Delegate research tasks to sub-agents using the task() tool
4. **Synthesize**: Review all sub-agent findings and consolidate citations
5. **Write Report**: Write a comprehensive final report to `/final_report.md`
6. **Verify**: Read `/research_request.md` and confirm you've addressed all aspects

## Research Planning Guidelines
- Batch similar research tasks into a single TODO
- For simple fact-finding, use 1 sub-agent
- For comparisons or multi-faceted topics, delegate to multiple parallel sub-agents

## Report Writing Guidelines
- Use clear section headings
- Write in paragraph form
- Cite sources inline using [1], [2], [3] format
- End with ### Sources section
"""

# 研究员指令：定义子智能体的研究行为
RESEARCHER_INSTRUCTIONS = """You are a research assistant conducting research on the user's input topic.

Your job is to use tools to gather information about the user's input topic.
You can use the tavily_search tool to find resources that can help answer the research question.

Follow these steps:
1. **Read the question carefully**
2. **Start with broader searches**
3. **After each search, pause and assess**
4. **Execute narrower searches as you gather information**
5. **Stop when you can answer confidently**

Tool Call Budgets:
- Simple queries: 2-3 searches max
- Complex queries: 5 searches max

When providing findings, structure with clear headings, cite sources inline, and include ### Sources.
"""

# 子智能体委派指令：定义何时以及如何并行委派任务
SUBAGENT_DELEGATION_INSTRUCTIONS = """# Sub-Agent Research Coordination

Your role is to coordinate research by delegating tasks from your TODO list to sub-agents.

## Delegation Strategy
- DEFAULT: Start with 1 sub-agent for most queries
- ONLY parallelize when the query requires comparison or has independent aspects

## Key Principles
- Bias towards single sub-agent
- Avoid premature decomposition
- Parallelize only for clear comparisons

## Parallel Execution Limits
- Use at most {max_concurrent_research_units} parallel sub-agents per iteration
"""

print("提示模板定义完成")

## 5. 创建智能体

   整合所有组件并创建深度研究智能体。支持多种模型提供商。

In [ ]:
from datetime import datetime

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

# 配置并发研究参数
max_concurrent_research_units = 3
max_researcher_iterations = 3

# 获取当前日期
current_date = datetime.now().strftime("%Y-%m-%d")

# 构建组合指令
INSTRUCTIONS = (
    RESEARCH_WORKFLOW_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
        max_concurrent_research_units=max_concurrent_research_units,
        max_researcher_iterations=max_researcher_iterations,
    )
)

# 定义研究子智能体
research_sub_agent = {
    "name": "research-agent",
    "description": "将研究任务委派给子智能体。一次给一个主题。",
    "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
    "tools": [tavily_search],
}

# 初始化模型（选择以下一种）
# Claude
# model = init_chat_model(model="anthropic:claude-sonnet-4-6", temperature=0.0)

# Gemini
# model = init_chat_model(model="google_genai:gemini-3.5-flash", temperature=0.0)

# OpenAI
# model = init_chat_model(model="openai:gpt-5.4", temperature=0.0)

# 创建深度智能体
agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=INSTRUCTIONS,
    subagents=[research_sub_agent],
)

print("深度研究智能体创建完成")
print(f"并发的子智能体数: {max_concurrent_research_units}")
print(f"最大研究轮次: {max_researcher_iterations}")

## 6. 运行研究任务

   向智能体提交一个研究查询，观察它如何规划、委派并综合结果。

In [ ]:
# 示例：运行深度研究任务
# 注意：实际运行时需要有效的 API 密钥

# input_message = {
#     "role": "user",
#     "content": "Research the latest trends in AI agent development in 2025"
# }

# for step in agent.stream(
#     {"messages": [input_message]},
#     stream_mode="updates",
# ):
#     for _, update in step.items():
#         if update and (messages := update.get("messages")):
#             for message in messages:
#                 message.pretty_print()

print("运行研究任务（API 密钥有效时取消注释）")

## 7. 关键设计原则

    | 原则 | 说明 |
    |------|------|
    | 偏向单子智能体 | 一个综合研究任务比多个窄任务更节省 Token |
    | 避免过早分解 | 不要将"研究 X"拆分为多个子任务——用 1 个子智能体覆盖所有方面 |
    | 仅在明确比较时并行 | 对比不同实体或地理分隔的数据时才使用多子智能体 |
    | 每次搜索后评估 | 检查已找到什么、还缺什么、是否足够回答 |
    | 工具调用预算 | 简单查询最多 2-3 次搜索，复杂查询最多 5 次 |

---

**参考链接**
- [LangChain Docs — Deep Agents Overview](https://docs.langchain.com/deep-agents)
- [Tavily Search API](https://tavily.com)
- [LangSmith Tracing](https://smith.langchain.com)
- [Deep Agents GitHub](https://github.com/langchain-ai/deep-agents)